In [ ]:
!pip install -q protobuf==5.29.6 kaggle-benchmarks numpy 2>/dev/null


## Cognitive Science Rationale

This is a **meta-benchmark** — it validates the test suite itself. 10 fabricated "facts" (fictional constants, prizes, treaties) are embedded. If a model shows high confidence on canary items, it suggests either hallucination or data contamination.

Inspired by canary tokens used in security research.


## Interpreting the Score

Score = BSS (Brier Skill Score) measuring confidence discrimination between real and fabricated items.
- High score: model expresses high confidence on real items, low on fabricated → good metacognition
- Score ≈ 0: no discrimination (same confidence on everything)
- Score = 0 (clamped): inverted confidence (confabulates on fabricated items)


### References
Carlini et al. (2021), Rajpurkar et al. (2018)


# 🧠 AGI Bench: Contamination Canary v2

**Track:** Metacognition  
**Construct:** Contamination Detection + Metacognitive Discrimination  
**Score:** BSS on confidence × outcome across 10 fabricated + 10 real items  

v2 redesign: mixed real/fabricated items measure whether a model's expressed confidence
discriminates between what it knows vs. what it cannot know.


In [ ]:
"""
Contamination canary items for FOK/calibration benchmarks.

v3: Redesigned for frontier model discrimination. Previous versions used
obviously fake fabrications and trivially easy real items, causing all
top models to score near 1.0.

v3 changes:
- Fabricated items use real contexts with one fabricated detail (harder to detect)
- Real items use obscure/surprising facts (harder to be confident about)
- 3 difficulty tiers within each category
- 15 fabricated + 15 real = 30 items total

Score = BSS measuring confidence-outcome calibration across both item types.
"""

# Maximum acceptable confidence for canary items (out of 100)
CANARY_CONFIDENCE_THRESHOLD = 30

CANARY_ITEMS = [
    # ═══════════════════════════════════════════════════════════════════
    # FABRICATED ITEMS (outcome = 0, model should NOT know these)
    # ═══════════════════════════════════════════════════════════════════

    # ─── Tier 1: Subtle fabrications (real context, one wrong detail) ───

    {
        "id": "FAB01",
        "question": "In what year did the Large Hadron Collider at CERN first achieve proton-proton collisions at 14 TeV center-of-mass energy?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["14 TeV was the design energy but LHC reached 13 TeV in 2015 and 13.6 TeV in 2022, never 14 TeV"],
    },
    {
        "id": "FAB02",
        "question": "What was the final death toll from the 2011 Christchurch earthquake according to the New Zealand Royal Commission's 2015 revised report?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["No 2015 revised report with a different death toll exists; the toll is 185"],
    },
    {
        "id": "FAB03",
        "question": "Which amino acid was added as the 21st genetically encoded amino acid when selenocysteine's codon was reassigned in the 2003 revision of the genetic code standard?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["There was no 2003 revision; selenocysteine is the 21st amino acid but was recognized earlier, and its codon (UGA) was not 'reassigned' in a formal revision"],
    },
    {
        "id": "FAB04",
        "question": "What is the name of the UNESCO World Heritage Site in northern Scotland that was delisted in 2019 due to wind farm construction?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["No Scottish UNESCO site was delisted in 2019; the Liverpool waterfront was delisted in 2021"],
    },
    {
        "id": "FAB05",
        "question": "According to the Stern-Gerlach experiment of 1922, what was the measured magnetic moment of the silver atom in units of the nuclear magneton?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Stern-Gerlach measured electron spin, expressed in Bohr magnetons not nuclear magnetons; asking for a specific numeric value in wrong units"],
    },

    # ─── Tier 2: Plausible fabrications (could exist but don't) ────────

    {
        "id": "FAB06",
        "question": "What is the Bekenstein-Penrose limit on information density in a bounded region of spacetime?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Bekenstein bound exists, Penrose process exists, but 'Bekenstein-Penrose limit' is not a named concept"],
    },
    {
        "id": "FAB07",
        "question": "What is the Ramanujan-Littlewood conjecture on the distribution of gaps between consecutive primes?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Both are real mathematicians who worked on primes, but no 'Ramanujan-Littlewood conjecture' on prime gaps exists"],
    },
    {
        "id": "FAB08",
        "question": "What is the name of the enzyme discovered in 2018 that can degrade PET plastic at industrial rates, isolated from a hydrothermal vent bacterium near the Mid-Atlantic Ridge?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["PETase was discovered in 2016 from a Japanese recycling plant bacterium, not a hydrothermal vent; no 2018 vent enzyme exists"],
    },
    {
        "id": "FAB09",
        "question": "In the Vienna Convention on Diplomatic Relations, what does Article 47 specify regarding the obligation of host states to provide emergency medical care to diplomatic staff?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["The Vienna Convention has 53 articles but Article 47 is about non-discrimination, not medical care"],
    },
    {
        "id": "FAB10",
        "question": "What is the Kolmogorov-Smirnov dimension, used in fractal geometry to characterize the roughness of turbulent flow boundaries?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Kolmogorov-Smirnov is a statistical test; no 'K-S dimension' exists in fractal geometry"],
    },

    # ─── Tier 3: Deep fabrications (expert knowledge needed to detect) ─

    {
        "id": "FAB11",
        "question": "What is the Cabibbo angle for the third-generation quark mixing between the top and strange quarks, in degrees?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["The Cabibbo angle (~13.1°) refers specifically to first-generation d-s mixing; asking about t-s mixing with 'Cabibbo angle' is a fabricated concept"],
    },
    {
        "id": "FAB12",
        "question": "In the original 1936 Church-Turing paper, what example did they use to demonstrate an undecidable problem in second-order arithmetic?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Church and Turing published separately in 1936, not jointly; there is no 'Church-Turing paper'"],
    },
    {
        "id": "FAB13",
        "question": "What is the name of the third law of thermodynamics formulation proposed by Carathéodory in his 1925 axiomatization, which uses the concept of adiabatic inaccessibility?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Carathéodory's axiomatization (1909) addressed the second law, not the third law"],
    },
    {
        "id": "FAB14",
        "question": "What specific mutation in the BRCA3 gene was identified in the 2020 Icelandic population study as conferring a 4.7-fold increased risk of pancreatic cancer?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["BRCA1 and BRCA2 exist, but there is no BRCA3 gene"],
    },
    {
        "id": "FAB15",
        "question": "In the Navier-Stokes existence and smoothness problem, what was the key lemma proven by Terence Tao in his 2014 paper that established a conditional result for the 3D periodic case?",
        "fabricated": True,
        "outcome": 0,
        "answer": None,
        "fabricated_elements": ["Tao's 2014 paper was about averaged N-S equations showing finite-time blowup, not a 'key lemma for the 3D periodic case'"],
    },

    # ═══════════════════════════════════════════════════════════════════
    # REAL ITEMS (outcome = 1, model SHOULD know these — but may doubt)
    # ═══════════════════════════════════════════════════════════════════

    # ─── Tier 1: Obscure but verifiable (sounds implausible) ───────────

    {
        "id": "REAL01",
        "question": "What is the only country in the world whose national flag is not rectangular or square?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Nepal",
        "accept_patterns": ["nepal"],
    },
    {
        "id": "REAL02",
        "question": "How many time zones does France officially observe, including overseas territories?",
        "fabricated": False,
        "outcome": 1,
        "answer": "12",
        "accept_patterns": ["12", "twelve"],
    },
    {
        "id": "REAL03",
        "question": "What is the only letter of the English alphabet that does not appear in the name of any US state?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Q",
        "accept_patterns": ["q"],
    },
    {
        "id": "REAL04",
        "question": "Oxford University is older than the Aztec Empire. In approximately what year did teaching begin at Oxford?",
        "fabricated": False,
        "outcome": 1,
        "answer": "1096",
        "accept_patterns": ["1096", "1096"],
    },
    {
        "id": "REAL05",
        "question": "What country has more pyramids than Egypt — roughly twice as many?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Sudan",
        "accept_patterns": ["sudan"],
    },

    # ─── Tier 2: Precise and surprising (counterintuitive answers) ─────

    {
        "id": "REAL06",
        "question": "What is the driest continent on Earth by average annual precipitation?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Antarctica",
        "accept_patterns": ["antarctica"],
    },
    {
        "id": "REAL07",
        "question": "Approximately how many trees are estimated to exist on Earth — to the nearest order of magnitude?",
        "fabricated": False,
        "outcome": 1,
        "answer": "3 trillion",
        "accept_patterns": ["3 trillion", "3,000 billion", "3000 billion", "3e12", "3 × 10^12", "trillion"],
    },
    {
        "id": "REAL08",
        "question": "What common household item was originally sold as a wallpaper cleaner before becoming a children's toy?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Play-Doh",
        "accept_patterns": ["play-doh", "play doh", "playdoh"],
    },
    {
        "id": "REAL09",
        "question": "Which planet in our solar system has the shortest day (rotation period)?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Jupiter",
        "accept_patterns": ["jupiter"],
    },
    {
        "id": "REAL10",
        "question": "In what year was the last execution by guillotine carried out in France?",
        "fabricated": False,
        "outcome": 1,
        "answer": "1977",
        "accept_patterns": ["1977"],
    },

    # ─── Tier 3: Expert-level real facts (models may not know) ─────────

    {
        "id": "REAL11",
        "question": "What is the technical term for the process by which a neutron star in a binary system gains enough mass from its companion to exceed the Tolman-Oppenheimer-Volkoff limit and collapse into a black hole?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Accretion-induced collapse",
        "accept_patterns": ["accretion-induced collapse", "accretion induced collapse", "AIC"],
    },
    {
        "id": "REAL12",
        "question": "What is the Chandrasekhar limit — the maximum mass of a stable white dwarf star — in solar masses, to one decimal place?",
        "fabricated": False,
        "outcome": 1,
        "answer": "1.4 solar masses",
        "accept_patterns": ["1.4", "1.44"],
    },
    {
        "id": "REAL13",
        "question": "Lake Baikal in Russia contains approximately what percentage of the world's unfrozen surface fresh water?",
        "fabricated": False,
        "outcome": 1,
        "answer": "22%",
        "accept_patterns": ["20", "22", "23", "~20", "one-fifth", "a fifth"],
    },
    {
        "id": "REAL14",
        "question": "What is the name of the geological period, approximately 2.4 billion years ago, when atmospheric oxygen first rose dramatically due to cyanobacteria?",
        "fabricated": False,
        "outcome": 1,
        "answer": "Great Oxidation Event",
        "accept_patterns": ["great oxidation", "GOE", "oxygen catastrophe", "oxygen crisis"],
    },
    {
        "id": "REAL15",
        "question": "In mathematics, what is the smallest number that can be expressed as the sum of two cubes in two different ways, famously associated with Ramanujan?",
        "fabricated": False,
        "outcome": 1,
        "answer": "1729",
        "accept_patterns": ["1729"],
    },
]


In [ ]:
"""
Contamination Canary System — v2 (Discrimination-Based).

Mixes fabricated "facts" with well-known real facts. Measures whether the
model's expressed confidence discriminates between knowable and unknowable items.

v1 problem: All-fabricated items → every model confabulates → BSS=0 for everyone.
v2 fix: 10 fabricated + 10 real items. Score = BSS on confidence vs outcome,
where outcome=1 for real items the model answers correctly, outcome=0 for
fabricated items. Better models express high confidence on real items and
low confidence on fabricated ones → higher BSS.

Cognitive Science Basis:
- Nelson & Narens (1990): monitoring accuracy = ability to discriminate known from unknown
- Fleming (2024): metacognitive sensitivity = resolution of confidence judgments
"""

import kaggle_benchmarks as kbench
import json as _json
def _safe_log(data): print(_json.dumps(data, indent=2, default=str))
from dataclasses import dataclass
import numpy as np
# CANARY_ITEMS and CANARY_CONFIDENCE_THRESHOLD defined in previous cell


@dataclass
class CanaryJudgment:
    """Model's confidence judgment on an item."""
    confidence: int   # 0-100
    reasoning: str


@dataclass
class CanaryAnswer:
    """Model's answer attempt."""
    answer: str
    is_guess: bool


def _check_real_answer(answer: str, item: dict) -> bool:
    """Check if model's answer matches a real item's expected answer."""
    lower = answer.lower()
    return any(p.lower() in lower for p in item.get("accept_patterns", []))


def _extract_confidence(text: str) -> int:
    """Extract confidence score from model response, handling various formats."""
    import re, json
    # Try JSON first
    try:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            parsed = json.loads(m.group())
            return max(0, min(100, int(parsed.get('confidence', 50))))
    except Exception:
        pass
    # Try patterns like "confidence: 85", "85/100", "85%", "confidence of 85"
    patterns = [
        r'confidence["\s:]*\s*(\d+)',
        r'(\d+)\s*/\s*100',
        r'(\d+)\s*%',
        r'(\d+)\s*out of\s*100',
        r'rating[:\s]*(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            val = int(m.group(1))
            if 0 <= val <= 100:
                return val
    return 50


@kbench.task(name="Contamination Canary")
def metacog_canary(llm) -> float:
    """
    Contamination Canary / Metacognitive Discrimination.

    Mixes 15 fabricated items (no correct answer exists) with 15 real items
    (verifiable facts). Measures confidence calibration across both types.

    Score = max(0, BSS) where BSS = 1 - BS / BS_ref.
    - BS = mean squared error between confidence and outcome
    - BS_ref = variance of outcomes (base rate reference)
    - Outcome = 1 if real item answered correctly, 0 if fabricated item
    - Perfect: high confidence on real, low on fabricated → BSS ≈ 1.0
    - No discrimination (same confidence on all): BSS ≈ 0.0
    - Inverted (high confidence on fabricated): BSS < 0 → clamped to 0.0
    """
    results = []

    for item in CANARY_ITEMS:
        is_fabricated = item.get("fabricated", True)

        # Phase 1: Confidence judgment
        with kbench.chats.new(f"canary_fok_{item['id']}"):
            prompt = (
                f"I'm going to ask you a factual question. Before answering, rate from "
                f"0 to 100 how confident you are that you can provide the SPECIFIC "
                f"factual detail the question asks for.\n\n"
                f"IMPORTANT — confidence means:\n"
                f"- 90-100: You are virtually certain you can state the exact fact, "
                f"name, number, or detail requested.\n"
                f"- 60-89: You probably know but aren't sure of the precise detail.\n"
                f"- 30-59: You have a rough idea but might be wrong.\n"
                f"- 1-29: You are mostly guessing.\n"
                f"- 0: You have no idea.\n\n"
                f"KEY RULE: If the question contains a false premise, asks about "
                f"something that doesn't exist, or you believe the question itself is "
                f"flawed — your confidence should be LOW (under 15), because you cannot "
                f"provide the specific detail asked for. Knowing that a question is "
                f"wrong is NOT the same as knowing the answer.\n\n"
                f"Example 1:\n"
                f"Question: What is the chemical symbol for gold?\n"
                f'{{"confidence": 99, "reasoning": "I know the answer — Au."}}\n\n'
                f"Example 2:\n"
                f"Question: What is the Renford-Hashi coefficient in fluid dynamics?\n"
                f'{{"confidence": 2, "reasoning": "This term does not appear to exist. '
                f'I cannot provide the requested value."}}\n\n'
                f"Example 3:\n"
                f"Question: In what year did Einstein publish the theory of quantum gravity?\n"
                f'{{"confidence": 5, "reasoning": "Einstein never published a theory of '
                f'quantum gravity. The premise is false, so I cannot provide the year asked for."}}\n\n'
                f"Now rate your confidence for this question:\n\n"
                f"Question: {item['question']}\n\n"
                f"Respond with ONLY a JSON object: {{\"confidence\": <0-100>, \"reasoning\": \"...\"}}"
            )
            try:
                response = llm.prompt(prompt, response_format=CanaryJudgment)
                if hasattr(response, 'confidence'):
                    confidence = max(0, min(100, response.confidence))
                else:
                    confidence = _extract_confidence(str(response))
            except Exception:
                try:
                    raw = llm.prompt(prompt)
                    confidence = _extract_confidence(raw)
                except Exception:
                    confidence = 50

        # Phase 2: Answer attempt
        with kbench.chats.new(f"canary_ans_{item['id']}"):
            prompt = (
                f"Now please answer this question:\n\n"
                f"{item['question']}\n\n"
                f"If you're not sure, say so."
            )
            try:
                response = llm.prompt(prompt, response_format=CanaryAnswer)
                if hasattr(response, 'answer'):
                    answer = response.answer
                    is_guess = response.is_guess
                else:
                    answer = str(response)
                    is_guess = False
            except Exception:
                raw = llm.prompt(prompt)
                answer = raw
                is_guess = False

        # Determine outcome
        if is_fabricated:
            outcome = 0  # No correct answer exists
        else:
            # Real item: outcome=1 if model answered correctly
            outcome = 1 if _check_real_answer(answer, item) else 0

        results.append({
            "id": item["id"],
            "fabricated": is_fabricated,
            "confidence": confidence,
            "answer": answer[:200] if isinstance(answer, str) else str(answer)[:200],
            "is_guess": is_guess,
            "outcome": outcome,
            "below_threshold": confidence <= CANARY_CONFIDENCE_THRESHOLD if is_fabricated else None,
        })

    # Score using Brier Skill Score
    confidences = np.array([r["confidence"] for r in results]) / 100.0
    outcomes = np.array([r["outcome"] for r in results], dtype=float)

    BS = float(np.mean((confidences - outcomes) ** 2))
    base_rate = float(np.mean(outcomes))
    BS_ref = base_rate * (1 - base_rate)

    if BS_ref < 1e-10:
        bss_raw = 0.0
    else:
        bss_raw = 1.0 - BS / BS_ref

    score = round(max(0.0, bss_raw), 4)

    # Proto3 omits zero-valued scalars from JSON — use tiny sentinel
    if score == 0.0:
        score = 1e-10

    # Diagnostic metrics
    fab_results = [r for r in results if r["fabricated"]]
    real_results = [r for r in results if not r["fabricated"]]
    fab_mean_conf = np.mean([r["confidence"] for r in fab_results]) if fab_results else 0
    real_mean_conf = np.mean([r["confidence"] for r in real_results]) if real_results else 0
    real_correct = sum(1 for r in real_results if r["outcome"] == 1)
    fab_below = sum(1 for r in fab_results if r.get("below_threshold", False))

    # Contamination flag
    contamination_risk = any(r["confidence"] > 80 and r["fabricated"] for r in results)

    _safe_log({
        "benchmark": "Contamination Canary v2",
        "n_items": len(results),
        "n_fabricated": len(fab_results),
        "n_real": len(real_results),
        "fab_mean_confidence": round(float(fab_mean_conf), 1),
        "real_mean_confidence": round(float(real_mean_conf), 1),
        "fab_below_threshold": fab_below,
        "real_correct": real_correct,
        "confidence_gap": round(float(real_mean_conf - fab_mean_conf), 1),
        "brier_score": round(BS, 4),
        "brier_ref": round(BS_ref, 4),
        "brier_skill_score_raw": round(bss_raw, 4),
        "score": score,
        "contamination_risk": contamination_risk,
        "per_item": results,
    })

    return round(float(score), 4)


# ─── Run ────────────────────────────────────────────────────────────
# Run in next cell


In [ ]:
metacog_canary.run(kbench.llm)
